In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [71]:
df = pd.read_json("../../../data/post_quality/raw/post_data_v4.json")
df.reset_index(drop=True)
df.head()

,id,title,body,effort,openness,is_confident,subreddit,tag
0,1,If your first-ever attempt at gambling went co...,,0,0,True,r/Showerthoughts,NaN
1,2,"According to birds, all land animals are botto...",,0,0,True,r/Showerthoughts,NaN
2,3,Very few people can actually keep their watch ...,,0,0,True,r/Showerthoughts,NaN
3,4,"What is keeping the really deadly diseases, li...",,0,0,True,r/askscience,NaN
4,5,When did humans start living indoors?,,0,0,True,r/askscience,NaN


In [72]:
## Reference for feature_engineering_V1 for other experiments

import re

def openness_feature_engineer_v1(df):
    df = df.copy()

    # Drop unwanted columns (ignore if missing)
    df.drop(columns=['tag', 'effort', 'id', 'is_confident'], inplace=True, errors='ignore')

    # Combine text fields
    df["text_for_feature"] = df["title"].fillna("") + " " + df["body"].fillna("") 
    df["text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.lower()

    # --- Question features ---
    def has_question_mark(text):
        if not isinstance(text, str):
            return 0
        return int("?" in text)

    def num_questions(text):
        if not isinstance(text, str):
            return 0
        return text.count("?")

    df["has_question_mark"] = df["text"].apply(has_question_mark)
    df["num_questions"] = df["text"].apply(num_questions)

    # --- Uncertainty features ---
    UNCERTAINTY_MARKERS = [
        "maybe", "might", "not sure", "i think", "i feel",
        "i guess", "i'm unsure", "i am unsure",
        "could be wrong", "not certain", "possibly"
    ]

    def uncertainty_count(text):
        if not isinstance(text, str):
            return 0
        return sum(1 for m in UNCERTAINTY_MARKERS if m in text)

    def has_uncertainty_marker(text):
        return int(uncertainty_count(text) > 0)

    df["uncertainty_count"] = df["text"].apply(uncertainty_count)
    df["has_uncertainty_marker"] = df["text"].apply(has_uncertainty_marker)

    # --- Invitation features ---
    INVITATION_PHRASES = [
        "what do you think",
        "any advice",
        "any thoughts",
        "can someone explain",
        "help me understand",
        "thoughts?",
        "does anyone know",
        "am i missing something"
    ]

    def has_invitation_phrase(text):
        if not isinstance(text, str):
            return 0
        return int(any(p in text for p in INVITATION_PHRASES))

    df["has_invitation_phrase"] = df["text"].apply(has_invitation_phrase)

    # --- Assertion features ---
    ASSERTION_MARKERS = [
        "obviously",
        "everyone knows",
        "the truth is",
        "there is no doubt",
        "clearly",
        "this proves",
        "it's obvious",
        "without question"
    ]

    def assertion_marker_count(text):
        if not isinstance(text, str):
            return 0
        return sum(1 for m in ASSERTION_MARKERS if m in text)

    df["assertion_marker_count"] = df["text"].apply(assertion_marker_count)

    # --- Reflective features ---
    REFLECTIVE_PHRASES = [
        "i'm trying to understand",
        "i am trying to understand",
        "i want to understand",
        "i'm learning",
        "i am learning",
        "i'm confused",
        "i am confused",
        "i don't fully understand",
        "i do not fully understand"
    ]

    def has_reflective_phrase(text):
        if not isinstance(text, str):
            return 0
        return int(any(p in text for p in REFLECTIVE_PHRASES))

    df["has_reflective_phrase"] = df["text"].apply(has_reflective_phrase)

    # Drop intermediate feature
    df.drop(columns=['num_questions'], inplace=True)

    return df

In [73]:
df = openness_feature_engineer_v1(df)

In [74]:
df.head()

,title,body,openness,subreddit,text_for_feature,text,has_question_mark,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase
0,If your first-ever attempt at gambling went co...,,0,r/Showerthoughts,If your first-ever attempt at gambling went co...,if your first-ever attempt at gambling went co...,0,0,0,0,0,0
1,"According to birds, all land animals are botto...",,0,r/Showerthoughts,"According to birds, all land animals are botto...","according to birds, all land animals are botto...",0,0,0,0,0,0
2,Very few people can actually keep their watch ...,,0,r/Showerthoughts,Very few people can actually keep their watch ...,very few people can actually keep their watch ...,0,0,0,0,0,0
3,"What is keeping the really deadly diseases, li...",,0,r/askscience,"What is keeping the really deadly diseases, li...","what is keeping the really deadly diseases, li...",1,0,0,0,0,0
4,When did humans start living indoors?,,0,r/askscience,When did humans start living indoors?,when did humans start living indoors?,1,0,0,0,0,0


### Some helper functions

In [75]:
## Let's build a helper function to make error_df
import pandas as pd

def build_error_df(df_test, y_true, y_pred, model_name):
    out = df_test.copy()
    out['true_label'] = y_true
    out['pred_label'] = y_pred
    out['correct'] = y_true == y_pred
    out['model'] = model_name
    return out

In [76]:
## Generic function to split df in same test and train
from sklearn.model_selection import train_test_split


def split_df(
    df,
    label_col,
    test_size=0.2,
    random_state=52,
    stratify=True
):
    """
    Splits dataframe into train and test sets.

    Returns
    -------
    df_train : pd.DataFrame
    df_test : pd.DataFrame
    """

    df = df.reset_index(drop=True)

    stratify_col = df[label_col] if stratify else None

    train_idx, test_idx = train_test_split(
        df.index,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_col
    )

    df_train = df.loc[train_idx]
    df_test = df.loc[test_idx]

    return df_train, df_test


In [77]:
import pandas as pd


def aggregate_fold_features(
    fold_features,
    label_key,
    top_k=10
):
    """
    Aggregates TF-IDF features across folds.

    Parameters
    ----------
    fold_features : list of dicts
        Each dict contains label_0_features / label_1_features DataFrames
    label_key : str
        Either "label_0_features" or "label_1_features"
    """

    rows = []

    for fold_data in fold_features:
        df = fold_data[label_key].copy()
        rows.append(df)

    all_features = pd.concat(rows)

    aggregated = (
        all_features
        .groupby("feature")
        .agg(
            freq=("feature", "count"),
            mean_coef=("coefficient", "mean"),
            mean_abs_coef=("coefficient", lambda x: x.abs().mean())
        )
        .sort_values(
            by=["freq", "mean_abs_coef"],
            ascending=[False, False]
        )
        .head(top_k)
        .reset_index()
    )

    return aggregated


In [78]:
def extract_top_tfidf_features(model, vectorizer, top_k=10):
    feature_names = vectorizer.get_feature_names_out()
    coefs = model.coef_[0]  # shape (n_features,)

    top_pos_idx = coefs.argsort()[-top_k:][::-1]
    top_neg_idx = coefs.argsort()[:top_k]

    label_1 = pd.DataFrame({
        "feature": feature_names[top_pos_idx],
        "coefficient": coefs[top_pos_idx]
    })

    label_0 = pd.DataFrame({
        "feature": feature_names[top_neg_idx],
        "coefficient": coefs[top_neg_idx]
    })

    return {
        "label_1": label_1,
        "label_0": label_0
    }


def extract_top_tfidf_features_for_mix_model(
    model,
    vectorizer,
    num_features_count,
    top_k=10
):
    feature_names = vectorizer.get_feature_names_out()

    # FULL coefficient vector
    coefs = model.coef_[0]

    # Slice only TF-IDF part
    tfidf_coefs = coefs[num_features_count:]

    # Safety check
    assert len(tfidf_coefs) == len(feature_names)

    top_pos_idx = tfidf_coefs.argsort()[-top_k:][::-1]
    top_neg_idx = tfidf_coefs.argsort()[:top_k]

    label_1 = pd.DataFrame({
        "feature": feature_names[top_pos_idx],
        "coefficient": tfidf_coefs[top_pos_idx]
    })

    label_0 = pd.DataFrame({
        "feature": feature_names[top_neg_idx],
        "coefficient": tfidf_coefs[top_neg_idx]
    })

    return {
        "label_1": label_1,
        "label_0": label_0
    }

In [79]:
def extract_top_mixed_features(
    model,
    vectorizer,
    num_feature_names,
    top_k=10
):
    tfidf_feature_names = vectorizer.get_feature_names_out()
    all_feature_names = np.concatenate(
        [num_feature_names, tfidf_feature_names]
    )

    coefs = model.coef_[0]
    assert len(coefs) == len(all_feature_names)

    top_pos_idx = coefs.argsort()[-top_k:][::-1]
    top_neg_idx = coefs.argsort()[:top_k]

    label_1 = pd.DataFrame({
        "feature": all_feature_names[top_pos_idx],
        "coefficient": coefs[top_pos_idx],
        "feature_type": [
            "numerical" if i < len(num_feature_names) else "tfidf"
            for i in top_pos_idx
        ]
    })

    label_0 = pd.DataFrame({
        "feature": all_feature_names[top_neg_idx],
        "coefficient": coefs[top_neg_idx],
        "feature_type": [
            "numerical" if i < len(num_feature_names) else "tfidf"
            for i in top_neg_idx
        ]
    })

    return {
        "label_1": label_1,
        "label_0": label_0
    }

In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


def run_tfidf_model(
    df_train,
    df_test,
    label_col,
    text_col="text_for_feature",
    vectorizer=None,
    model=None,
    model_name="tfidf_only"
):
    """
    Trains and evaluates a TF-IDF text classification model.

    Returns
    -------
    dict with:
        - model
        - vectorizer
        - classification_report
        - errors_df
        - accuracy
        - y_pred
    """

    if vectorizer is None:
        vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            max_df=0.95,
            min_df=5,
            stop_words="english"
        )

    if model is None:
        model = LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )

    # Split data
    X_train_text = df_train[text_col]
    X_test_text = df_test[text_col]

    y_train = df_train[label_col]
    y_test = df_test[label_col]

    # Vectorize text
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    # Train model
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Metrics
    report = classification_report(y_test, y_pred)

    errors_df = build_error_df(
        df_test=df_test,
        y_true=y_test,
        y_pred=y_pred,
        model_name=model_name
    )

    errors_df["error_type"] = ""
    errors_df["error_subtype"] = ""
    errors_df["notes"] = ""

    return {
        "model": model,
        "vectorizer": vectorizer,
        "classification_report": report,
        "errors_df": errors_df,
        "accuracy": accuracy_score(y_test, y_pred),
        "y_pred": y_pred
    }


In [81]:
from sklearn.model_selection import StratifiedKFold

def run_k_fold_tfidf_error_analysis(
    df: pd.DataFrame,
    n_splits=5,
    label_col="openness",
    text_col="text_for_feature",
    vectorizer=None,
    model=None,
    model_name="tfidf_only",
    top_k_features=10
):
    """
    Runs k-fold cross-validation error analysis for a TF-IDF text model.
    """

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    all_errors = []
    fold_features = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df[label_col])):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        result = run_tfidf_model(
            df_train=train_df,
            df_test=test_df,
            label_col=label_col,
            text_col=text_col,
            vectorizer=vectorizer,
            model=model,
            model_name=f"{model_name}_fold_{fold}"
        )

        # ----- errors -----
        errors_df = result["errors_df"].copy()
        errors_df["fold"] = fold
        all_errors.append(errors_df)

        # ----- features -----
        top_feats = extract_top_tfidf_features(
            result["model"],
            result["vectorizer"],
            top_k_features
        )

        fold_features.append({
            "label_1_features": top_feats["label_1"],
            "label_0_features": top_feats["label_0"]
        })

    # ----- error aggregation -----
    errors_all = pd.concat(all_errors).reset_index(drop=True)
    errors_only = errors_all[errors_all["correct"] == False]

    errors_all["error_type"] = "N/A"
    errors_all["error_subtype"] = "N/A"
    errors_all["notes"] = ""

    true_label_counts = errors_only["true_label"].value_counts()
    pred_label_counts = errors_only["pred_label"].value_counts()
    error_group_counts = (
        errors_only
        .groupby(["true_label", "pred_label"])
        .size()
    )

    # ----- feature aggregation -----
    stable_label_1_features = aggregate_fold_features(
        fold_features,
        label_key="label_1_features",
        top_k=top_k_features
    )

    stable_label_0_features = aggregate_fold_features(
        fold_features,
        label_key="label_0_features",
        top_k=top_k_features
    )

    # ----- common features (NEW) -----
    common_features = (
        set(stable_label_1_features["feature"]) &
        set(stable_label_0_features["feature"])
    )

    common_features_df = (
        stable_label_1_features
        .merge(
            stable_label_0_features,
            on="feature",
            suffixes=("_label_1", "_label_0")
        )
        .assign(is_common_feature=True)
    )

    return {
        "errors_all": errors_all,
        "errors_only": errors_only,
        "true_label_counts": true_label_counts,
        "pred_label_counts": pred_label_counts,
        "error_group_counts": error_group_counts,

        # Stable features
        "stable_features": {
            "label_1": stable_label_1_features,
            "label_0": stable_label_0_features
        },

        # ambiguous / shared features
        "common_features": {
            "feature_set": common_features,
            "feature_df": common_features_df
        }
    }


In [82]:
import numpy as np
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

def run_numerical_tfidf_model(
    df_train,
    df_test,
    label_col,
    text_col="text_for_feature",
    num_cols=None,
    scaler=None,
    vectorizer=None,
    model=None,
    model_name="numerical_tfidf"
):
    """
    Trains and evaluates a numerical + TF-IDF classification model.
    """

    if num_cols is None:
        num_cols = df_train.select_dtypes(include="number").columns.tolist()
        num_cols.remove(label_col)

    if scaler is None:
        scaler = StandardScaler()

    if vectorizer is None:
        vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=5,
            max_df=0.95,
            stop_words="english"
        )

    if model is None:
        model = LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )

    # ----- Numerical features -----
    X_train_num = scaler.fit_transform(df_train[num_cols])
    X_test_num = scaler.transform(df_test[num_cols])

    # ----- Text features -----
    X_train_text = vectorizer.fit_transform(df_train[text_col])
    X_test_text = vectorizer.transform(df_test[text_col])

    # ----- Concatenate -----
    X_train = hstack([X_train_num, X_train_text])
    X_test = hstack([X_test_num, X_test_text])

    y_train = df_train[label_col]
    y_test = df_test[label_col]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred)

    errors_df = build_error_df(
        df_test=df_test,
        y_true=y_test,
        y_pred=y_pred,
        model_name=model_name
    )

    errors_df["error_type"] = ""
    errors_df["error_subtype"] = ""
    errors_df["notes"] = ""

    return {
        "model": model,
        "scaler": scaler,
        "vectorizer": vectorizer,
        "classification_report": report,
        "accuracy": accuracy_score(y_test, y_pred),
        "errors_df": errors_df,
        "y_pred": y_pred
    }


# TF-IDF Only Model

In [83]:
result = run_k_fold_tfidf_error_analysis(df=df,label_col='openness')


### Features that are consistently pushing towards label_1

In [84]:
result['stable_features']['label_1']

,feature,freq,mean_coef,mean_abs_coef
0,american,5,0.827229,0.827229
1,country,5,0.732313,0.732313
2,ai,5,0.704249,0.704249
3,people,4,0.814177,0.814177
4,dae,4,0.785703,0.785703
5,world,4,0.693406,0.693406
6,thoughts,3,0.706565,0.706565
7,war,3,0.668226,0.668226
8,social,3,0.665556,0.665556
9,society,3,0.632589,0.632589


### Features that are consistently pushing towards label_0

In [85]:
result['stable_features']['label_0']

,feature,freq,mean_coef,mean_abs_coef
0,got,5,-0.798576,0.798576
1,start,5,-0.725160,0.725160
2,really,4,-0.704169,0.704169
3,better,4,-0.625869,0.625869
4,looking,3,-0.701219,0.701219
5,dont,3,-0.684606,0.684606
6,working,3,-0.610217,0.610217
7,asked,3,-0.605228,0.605228
8,went,3,-0.591811,0.591811
9,use,2,-0.640420,0.640420


### Shared features


In [86]:
result['common_features']['feature_df']

,feature,freq_label_1,mean_coef_label_1,mean_abs_coef_label_1,freq_label_0,mean_coef_label_0,mean_abs_coef_label_0,is_common_feature


In [87]:
result['common_features']['feature_set']

set()

In [88]:
result['errors_only']

,title,body,openness,subreddit,text_for_feature,text,has_question_mark,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
0,Very few people can actually keep their watch ...,,0,r/Showerthoughts,Very few people can actually keep their watch ...,very few people can actually keep their watch ...,0,0,0,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
2,"Just realized I've been saying ""epitome"" wrong...","Been saying ""epi-TOME"" like it rhymes with hom...",0,r/self,"Just realized I've been saying ""epitome"" wrong...","just realized i've been saying ""epitome"" wrong...",0,0,0,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
4,DAE I fail to plan things and find I’m so inde...,I’d like to fit in a few things over the next ...,0,r/DoesAnybodyElse,DAE I fail to plan things and find I’m so inde...,dae i fail to plan things and find i’m so inde...,0,0,0,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
6,Legitimately dying alone,Yeah. I genuinely don't think its possible for...,0,r/lonely,Legitimately dying alone Yeah. I genuinely don...,legitimately dying alone yeah. i genuinely don...,0,0,0,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
7,People are making plans with money I don't hav...,So a while ago I was in a car accident. I'm no...,0,r/offmychest,People are making plans with money I don't hav...,people are making plans with money i don't hav...,0,1,1,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,Is it unethical to steal from large corporatio...,I've been having this debate with a few friend...,1,r/Ethics,Is it unethical to steal from large corporatio...,is it unethical to steal from large corporatio...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_4,,,,4
232,I'm a US sailor on a ship in the Pacific that ...,How quickly will be I be transferred to a new ...,1,r/AskHistorians,I'm a US sailor on a ship in the Pacific that ...,i'm a us sailor on a ship in the pacific that ...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_4,,,,4
235,CMV: Emergency sirens and car horns should not...,"First, some definitions\nBy “emergency sirens,...",1,r/changemyview,CMV: Emergency sirens and car horns should not...,cmv: emergency sirens and car horns should not...,0,1,1,0,1,0,1,0,False,tfidf_only_fold_4,,,,4
236,"If we understand how life and emotions work, w...",This is something I keep thinking about.\nAs a...,1,r/askphilosophy,"If we understand how life and emotions work, w...","if we understand how life and emotions work, w...",1,1,1,0,0,0,1,0,False,tfidf_only_fold_4,,,,4


In [89]:
sample = result['errors_only'].sample(25, random_state=42)


In [90]:
sample

,title,body,openness,subreddit,text_for_feature,text,has_question_mark,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
169,Is there a religion like this?,I've thought a lot about my religion lately an...,1,r/AskReligion,Is there a religion like this? I've thought a ...,is there a religion like this? i've thought a ...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_3,,,,3
61,the self checkout was broken and I almost crie...,I went to Target to grab some stuff and of cou...,0,r/socialanxiety,the self checkout was broken and I almost crie...,the self checkout was broken and i almost crie...,1,0,0,0,1,0,0,1,False,tfidf_only_fold_1,,,,1
7,People are making plans with money I don't hav...,So a while ago I was in a car accident. I'm no...,0,r/offmychest,People are making plans with money I don't hav...,people are making plans with money i don't hav...,0,1,1,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
25,What religion did Jesus and his disciples prac...,Weren't they all still practicing Judaism? And...,1,r/AskReligion,What religion did Jesus and his disciples prac...,what religion did jesus and his disciples prac...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_0,,,,0
115,Why don't kernel level anti-cheats exist on li...,Its my understanding that programs can get acc...,0,r/linuxquestions,Why don't kernel level anti-cheats exist on li...,why don't kernel level anti-cheats exist on li...,1,0,0,0,0,0,0,1,False,tfidf_only_fold_2,,,,2
152,"I hate parties, and nobody seems to understand.",I've been an introvert since birth and I've ne...,0,r/introvert,"I hate parties, and nobody seems to understand...","i hate parties, and nobody seems to understand...",0,1,1,0,0,0,0,1,False,tfidf_only_fold_3,,,,3
220,How do the gods (or the closest thing to them)...,Do they ignore them? Do they despise them? Do ...,1,r/worldbuilding,How do the gods (or the closest thing to them)...,how do the gods (or the closest thing to them)...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_4,,,,4
8,The snyderverse subreddit is like a cult.,Some of the rules saying you can’t disrespect ...,0,r/movies,The snyderverse subreddit is like a cult. Some...,the snyderverse subreddit is like a cult. some...,0,0,0,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
226,Do Americans mainly drink coffee without milk?,"In films, coffee is often depicted as being dr...",1,r/AskAnAmerican,Do Americans mainly drink coffee without milk?...,do americans mainly drink coffee without milk?...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_4,,,,4
37,How did traits and stories of gods like Odin o...,A few questions related to this as I try and w...,1,r/religious_studies,How did traits and stories of gods like Odin o...,how did traits and stories of gods like odin o...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_0,,,,0


In [91]:
result['errors_only']['openness'].value_counts()

openness
1    39
0    29
Name: count, dtype: int64

### Error Analysis – TF-IDF Baseline (Openness)

The TF-IDF-only baseline exhibits systematic failure modes that stem from its reliance on lexical patterns rather than intent understanding.

#### False Negatives (Open → Predicted Closed)
- Opinion- or belief-seeking questions (e.g., religion, philosophy) are often misclassified as closed.
- Hypothetical or exploratory questions lack strong lexical cues associated with openness.
- These posts are topic-heavy but do not contain explicit discussion markers, leading the model to treat them as factual queries.

#### False Positives (Closed → Predicted Open)
- Technical explanation-seeking questions (e.g., “Why does X work this way?”) are misclassified as open despite having constrained answer spaces.
- Personal rants or story-like posts are predicted as open due to expressive language and length, even when they do not invite discussion.

#### Conclusion
This behavior is expected for a TF-IDF-based model, which captures word frequency correlations but lacks semantic or intent-level understanding. These findings motivate the addition of lightweight numerical intent features to better distinguish open-ended discussion prompts from factual explanation queries.


# TF-IDF + Baseline Numerical Feature Model

In [92]:
def run_k_fold_numerical_tfidf_error_analysis(
    df: pd.DataFrame,
    n_splits=5,
    label_col="label",
    text_col="text_for_feature",
    num_cols=None,
    model_name="numerical_tfidf",
    top_k_features=10
):

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    all_errors = []
    fold_tfidf_features = []
    fold_mixed_features = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df[label_col])):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        result = run_numerical_tfidf_model(
            df_train=train_df,
            df_test=test_df,
            label_col=label_col,
            text_col=text_col,
            num_cols=num_cols,
            model_name=f"{model_name}_fold_{fold}"
        )

        # ----- errors -----
        errors_df = result["errors_df"].copy()
        errors_df["fold"] = fold
        all_errors.append(errors_df)

        # ----- TF-IDF ONLY features -----
        tfidf_feats = extract_top_tfidf_features_for_mix_model(
            model=result["model"],
            vectorizer=result["vectorizer"],
            num_features_count=len(num_cols),
            top_k=top_k_features
        )

        fold_tfidf_features.append({
            "label_1_features": tfidf_feats["label_1"],
            "label_0_features": tfidf_feats["label_0"]
        })

        # ----- MIXED features (NEW) -----
        mixed_feats = extract_top_mixed_features(
            model=result["model"],
            vectorizer=result["vectorizer"],
            num_feature_names=np.array(num_cols),
            top_k=top_k_features
        )

        fold_mixed_features.append({
            "label_1_features": mixed_feats["label_1"],
            "label_0_features": mixed_feats["label_0"]
        })

    # ----- error aggregation -----
    errors_all = pd.concat(all_errors).reset_index(drop=True)
    errors_only = errors_all[errors_all["correct"] == False]

    errors_all["error_type"] = "N/A"
    errors_all["error_subtype"] = "N/A"
    errors_all["notes"] = ""

    true_label_counts = errors_only["true_label"].value_counts()
    pred_label_counts = errors_only["pred_label"].value_counts()
    error_group_counts = (
        errors_only
        .groupby(["true_label", "pred_label"])
        .size()
    )

    # ----- TF-IDF feature aggregation -----
    stable_tfidf_label_1 = aggregate_fold_features(
        fold_tfidf_features,
        label_key="label_1_features",
        top_k=top_k_features
    )

    stable_tfidf_label_0 = aggregate_fold_features(
        fold_tfidf_features,
        label_key="label_0_features",
        top_k=top_k_features
    )

    # ----- MIXED feature aggregation -----
    stable_mixed_label_1 = aggregate_fold_features(
        fold_mixed_features,
        label_key="label_1_features",
        top_k=top_k_features
    )

    stable_mixed_label_0 = aggregate_fold_features(
        fold_mixed_features,
        label_key="label_0_features",
        top_k=top_k_features
    )

    return {
        "errors_all": errors_all,
        "errors_only": errors_only,
        "true_label_counts": true_label_counts,
        "pred_label_counts": pred_label_counts,
        "error_group_counts": error_group_counts,

        # TF-IDF-only stable features
        "stable_tfidf_features": {
            "label_1": stable_tfidf_label_1,
            "label_0": stable_tfidf_label_0
        },

        # MIXED stable features (NEW)
        "stable_mixed_features": {
            "label_1": stable_mixed_label_1,
            "label_0": stable_mixed_label_0
        }
    }

In [93]:
non_object_cols = df.select_dtypes(exclude=["object"]).columns.tolist()
non_object_cols.pop(0) # To remove the label column
non_object_cols

['has_question_mark',
 'uncertainty_count',
 'has_uncertainty_marker',
 'has_invitation_phrase',
 'assertion_marker_count',
 'has_reflective_phrase']

In [94]:
result_tfidf_numerical = run_k_fold_numerical_tfidf_error_analysis(df=df,num_cols=non_object_cols, label_col="openness")

No common feature

In [95]:
result_tfidf_numerical['stable_tfidf_features']['label_0']

,feature,freq,mean_coef,mean_abs_coef
0,really,4,-0.834598,0.834598
1,start,4,-0.693710,0.693710
2,dont,4,-0.677381,0.677381
3,looking,3,-0.691180,0.691180
4,talk,3,-0.660399,0.660399
5,im,3,-0.580333,0.580333
6,thank,3,-0.579552,0.579552
7,month,3,-0.559606,0.559606
8,got,2,-0.701249,0.701249
9,use,2,-0.648989,0.648989


In [96]:
result_tfidf_numerical['stable_tfidf_features']['label_1']

,feature,freq,mean_coef,mean_abs_coef
0,think,5,0.760097,0.760097
1,people,4,0.944164,0.944164
2,dae,4,0.671296,0.671296
3,culture,4,0.620123,0.620123
4,american,3,0.744884,0.744884
5,country,3,0.604384,0.604384
6,job,3,0.571103,0.571103
7,society,3,0.555053,0.555053
8,european,3,0.549324,0.549324
9,thoughts,2,0.613347,0.613347


In [97]:
result_tfidf_numerical['stable_mixed_features']['label_0']

,feature,freq,mean_coef,mean_abs_coef
0,really,4,-0.834598,0.834598
1,start,4,-0.693710,0.693710
2,dont,4,-0.677381,0.677381
3,looking,3,-0.691180,0.691180
4,talk,3,-0.660399,0.660399
5,im,3,-0.580333,0.580333
6,thank,3,-0.579552,0.579552
7,got,2,-0.701249,0.701249
8,use,2,-0.648989,0.648989
9,know,2,-0.615512,0.615512


In [98]:
result_tfidf_numerical['stable_mixed_features']['label_1']

,feature,freq,mean_coef,mean_abs_coef
0,has_question_mark,5,0.914707,0.914707
1,think,5,0.760097,0.760097
2,people,4,0.944164,0.944164
3,dae,4,0.671296,0.671296
4,american,3,0.744884,0.744884
5,culture,3,0.641901,0.641901
6,country,3,0.604384,0.604384
7,job,3,0.571103,0.571103
8,thoughts,2,0.613347,0.613347
9,society,2,0.589212,0.589212


## Observation and Hypothesis:

* TF-IDF features suggest that posts framed around generalized social concepts are more likely to be classified as open, whereas first-person experiential narration correlates with closed labels.
* By seeing mixed feature, I can see that other than `has_question_mark` numerical feature, other numerical feature play as a tie breaker role. 

In [104]:
sample_mix = result['errors_only'].sample(25, random_state=42)

In [105]:
sample_mix

,title,body,openness,subreddit,text_for_feature,text,has_question_mark,uncertainty_count,has_uncertainty_marker,has_invitation_phrase,assertion_marker_count,has_reflective_phrase,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
169,Is there a religion like this?,I've thought a lot about my religion lately an...,1,r/AskReligion,Is there a religion like this? I've thought a ...,is there a religion like this? i've thought a ...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_3,,,,3
61,the self checkout was broken and I almost crie...,I went to Target to grab some stuff and of cou...,0,r/socialanxiety,the self checkout was broken and I almost crie...,the self checkout was broken and i almost crie...,1,0,0,0,1,0,0,1,False,tfidf_only_fold_1,,,,1
7,People are making plans with money I don't hav...,So a while ago I was in a car accident. I'm no...,0,r/offmychest,People are making plans with money I don't hav...,people are making plans with money i don't hav...,0,1,1,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
25,What religion did Jesus and his disciples prac...,Weren't they all still practicing Judaism? And...,1,r/AskReligion,What religion did Jesus and his disciples prac...,what religion did jesus and his disciples prac...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_0,,,,0
115,Why don't kernel level anti-cheats exist on li...,Its my understanding that programs can get acc...,0,r/linuxquestions,Why don't kernel level anti-cheats exist on li...,why don't kernel level anti-cheats exist on li...,1,0,0,0,0,0,0,1,False,tfidf_only_fold_2,,,,2
152,"I hate parties, and nobody seems to understand.",I've been an introvert since birth and I've ne...,0,r/introvert,"I hate parties, and nobody seems to understand...","i hate parties, and nobody seems to understand...",0,1,1,0,0,0,0,1,False,tfidf_only_fold_3,,,,3
220,How do the gods (or the closest thing to them)...,Do they ignore them? Do they despise them? Do ...,1,r/worldbuilding,How do the gods (or the closest thing to them)...,how do the gods (or the closest thing to them)...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_4,,,,4
8,The snyderverse subreddit is like a cult.,Some of the rules saying you can’t disrespect ...,0,r/movies,The snyderverse subreddit is like a cult. Some...,the snyderverse subreddit is like a cult. some...,0,0,0,0,0,0,0,1,False,tfidf_only_fold_0,,,,0
226,Do Americans mainly drink coffee without milk?,"In films, coffee is often depicted as being dr...",1,r/AskAnAmerican,Do Americans mainly drink coffee without milk?...,do americans mainly drink coffee without milk?...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_4,,,,4
37,How did traits and stories of gods like Odin o...,A few questions related to this as I try and w...,1,r/religious_studies,How did traits and stories of gods like Odin o...,how did traits and stories of gods like odin o...,1,0,0,0,0,0,1,0,False,tfidf_only_fold_0,,,,0


## Experiment: TF-IDF + Numerical Features (Openness Classification)

### Objective
Evaluate whether combining surface-level numerical intent features with TF-IDF lexical representations reduces false positives observed in the numerical baseline while improving recall over the TF-IDF-only model.

---

### Model Setup
- Text Representation: TF-IDF (1–2 grams)
- Numerical Features:
  - question_mark_count
  - uncertainty_phrase_count
  - first_person_pronoun_ratio
  - text_length
- Classifier: Logistic Regression (class-balanced)
- Evaluation: 5-fold cross-validation with manual error analysis

---

### Observations
- Total errors: 67
- False positives: 54 (~80% of errors)
- False negatives primarily occur in belief-driven and hypothetical questions
- Numerical features dominate decision boundaries, particularly question-related signals

---

### Error Taxonomy

#### 1. Reflective / Personal / Venting Posts (False Positives)
Posts narrating personal experiences or expressing frustration are misclassified as open due to:
- First-person pronouns
- Abstract/emotional vocabulary
- Increased length

These posts are expressive but do not invite discussion, indicating a semantic intent mismatch.

#### 2. Factual or Technical Questions (False Positives)
Explanation-seeking questions (e.g., “Why does X work?”) are misclassified as open due to:
- Question marks
- Interrogative phrasing

Despite appearing inquisitive, these questions have constrained answer spaces.

#### 3. Hypothetical / Exploratory Questions (False Negatives)
Exploratory questions lack strong lexical or surface cues and are frequently misclassified as closed.

#### 4. Societal / Cultural / Religious Belief Questions (False Negatives)
Belief-oriented questions are downgraded due to neutral tone and lack of strong abstract markers, despite clear discussion-seeking intent.

---

### Conclusion
The mixed TF-IDF + numerical model reintroduces surface-level heuristics that increase false positives without resolving intent-level ambiguity. Error analysis shows that the remaining failure modes require semantic understanding of discourse intent rather than additional surface or structural features.

---

### Next Step
Transition to semantic intent modeling using sentence-level embeddings while minimizing handcrafted numerical features to prevent shortcut learning.

# Decision & Rationale

This experiment tested the hypothesis that augmenting **TF-IDF representations** with **surface-level numerical intent features** would reduce false positives observed in the numerical baseline while preserving the lexical strengths of TF-IDF.

## Outcome

**The hypothesis is falsified.**

Error analysis shows that numerical features—particularly **question-related** and **first-person cues**—reintroduce shortcut learning and significantly increase false positives. These features amplify superficial signals of curiosity or expressiveness without resolving intent-level ambiguity.

## Consequences

- **Handcrafted numerical openness features are intentionally deprecated** for subsequent models.
- **Future experiments will avoid surface heuristics** that correlate with openness labels but lack semantic grounding.
- **Remaining error modes require modeling discourse intent and answer-space openness**, which cannot be reliably captured via token frequency or structural cues alone.

## Conclusion

This experiment establishes a clear boundary: **further gains must come from semantic representations rather than feature engineering.**
